# Lab 01: Exploring Bedrock Foundation Models

## Overview
In this lab you will invoke multiple foundation models on Amazon Bedrock using both the InvokeModel and Converse APIs. You will compare outputs across providers, tune inference parameters, and understand the difference between on-demand and provisioned throughput.

## Learning Objectives
- Invoke foundation models using `InvokeModel` and `Converse` APIs via boto3
- Compare outputs from Claude, Titan, Llama, and Mistral on the same prompt
- Tune inference parameters: temperature, top_p, top_k, max_tokens, stop sequences
- Stream model responses using `ConverseStream`
- Understand on-demand vs provisioned throughput pricing and use cases

## Exam Domain
**Selection & Implementation of Foundation Models (26%)** — This lab covers Task 1.1 (select appropriate foundation models) and Task 1.2 (implement foundation models using AWS services).

## Architecture Diagram
![Lab 01 Architecture](../assets/diagrams/lab-01-bedrock-foundation-models.png)

In [ ]:
%pip install boto3 --quiet

In [ ]:
import boto3
import json
import os
import time

# us-east-1 has the broadest Bedrock model support across providers
REGION = "us-east-1"

# SageMaker Studio sets AWS_DEFAULT_REGION and mounts /opt/ml
if os.environ.get("AWS_DEFAULT_REGION") and os.path.exists("/opt/ml"):
    session = boto3.Session(region_name=REGION)
    print("Running in SageMaker Studio")
else:
    session = boto3.Session(region_name=REGION)
    print("Running locally")

# bedrock = control-plane client (list models, create jobs)
# bedrock-runtime = data-plane client (invoke models)
bedrock = session.client("bedrock")
bedrock_runtime = session.client("bedrock-runtime")

# Verify IAM permissions allow Bedrock access
models = bedrock.list_foundation_models()
print(f"Bedrock access confirmed. Found {len(models['modelSummaries'])} models available.")

## A. Listing Available Models

Amazon Bedrock is a fully managed service that provides access to foundation models from multiple providers through a single API. The available providers include:

- **Anthropic** — Claude family (text generation, analysis, coding)
- **Amazon** — Titan family (text, embeddings, image generation)
- **Meta** — Llama family (open-weight text generation)
- **Mistral AI** — Mistral and Mixtral (efficient text generation)
- **Cohere** — Command family (text generation, embeddings)
- **AI21 Labs** — Jamba family (text generation)
- **Stability AI** — Stable Diffusion (image generation)

Each model has different strengths, context window sizes, pricing, and supported features. The first step in working with Bedrock is understanding what models are available to you. Let's list them and group by provider.

In [ ]:
response = bedrock.list_foundation_models()
providers = {}
for model in response["modelSummaries"]:
    provider = model["providerName"]
    providers.setdefault(provider, []).append(model["modelId"])

for provider, model_ids in sorted(providers.items()):
    print(f"\n{provider} ({len(model_ids)} models):")
    for mid in model_ids[:3]:
        print(f"  - {mid}")
    if len(model_ids) > 3:
        print(f"  ... and {len(model_ids) - 3} more")

## B. InvokeModel API (Model-Specific Formats)

The `InvokeModel` API is the original way to call foundation models on Bedrock. It sends a raw JSON body and returns a raw JSON response, but **each provider has its own schema** -- you must know the exact format for every model you call.

> **Exam tip:** The exam tests whether you know the difference between `InvokeModel` (provider-specific) and `Converse` (unified). Sections B and C illustrate this side by side.

### Claude via InvokeModel

Anthropic Claude uses the **Messages API** format: `anthropic_version`, a `messages` array, and `max_tokens`. Response text is nested at `content[0].text`.

In [ ]:
prompt = "Explain what Amazon Bedrock is in two sentences."

response = bedrock_runtime.invoke_model(
    # Cross-region inference prefix "us." routes to the nearest US endpoint
    modelId="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    contentType="application/json",
    accept="application/json",
    body=json.dumps({
        # Required: locks to a stable API version for Anthropic models on Bedrock
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 256,
        # InvokeModel for Claude requires the Messages API structure
        "messages": [{"role": "user", "content": prompt}]
    })
)
# Claude nests output at content[0].text — different from every other provider
result = json.loads(response["body"].read())
print("Claude:", result["content"][0]["text"])

### Llama 3 via InvokeModel

Meta Llama uses a completely different schema: a flat `prompt` string (not a messages array), `max_gen_len` (not `max_tokens`), and returns text at `generation` (not `content[0].text`).

In [ ]:
response = bedrock_runtime.invoke_model(
    modelId="meta.llama3-8b-instruct-v1:0",
    contentType="application/json",
    accept="application/json",
    body=json.dumps({
        # Llama uses "prompt" (flat string), not "messages" (array)
        "prompt": prompt,
        # Llama calls it "max_gen_len", not "max_tokens"
        "max_gen_len": 256,
        "temperature": 0.7
    })
)
# Llama returns text at "generation", not "content[0].text"
result = json.loads(response["body"].read())
print("Llama 3:", result["generation"])

### The Problem with InvokeModel

The same task -- "send a prompt, get text back" -- required completely different code per provider:

- **Switching models** means rewriting integration code
- **Comparing models** side by side is impractical
- **Maintaining code** across provider version changes is error-prone

This is exactly the problem the **Converse API** solves.

## C. Converse API (Unified Interface)

The **Converse API** provides a single, consistent interface for invoking any text model on Bedrock -- one format, one response structure, regardless of provider.

| Feature | InvokeModel | Converse |
|---------|-------------|----------|
| Request format | Provider-specific JSON | Unified `messages` array |
| Response parsing | Varies by provider | Always `output.message.content[0].text` |
| Model switching | Rewrite integration code | Change only `modelId` |
| Streaming | `InvokeModelWithResponseStream` | `ConverseStream` |

```mermaid
flowchart LR
    subgraph InvokeModel ["InvokeModel (provider-specific)"]
        direction TB
        P1[Prompt] --> F1[Claude Format]
        P1 --> F2[Llama Format]
        P1 --> F3[Mistral Format]
    end
    subgraph Converse ["Converse API (unified)"]
        direction TB
        P2[Prompt] --> U[Unified messages array]
        U --> M1[Claude]
        U --> M2[Llama]
        U --> M3[Mistral]
    end
```

> **Exam tip:** The Converse API is the **recommended** interface for all new Bedrock applications. Expect questions asking which API provides a consistent interface across providers.

In [ ]:
# Converse API — the unified interface the exam expects you to know
def invoke_converse(model_id, prompt, max_tokens=256, temperature=0.7):
    response = bedrock_runtime.converse(
        modelId=model_id,
        # Same messages format regardless of provider
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": max_tokens, "temperature": temperature}
    )
    # Response path is always the same — no provider-specific parsing
    return response["output"]["message"]["content"][0]["text"]

models_to_compare = [
    ("us.anthropic.claude-sonnet-4-5-20250929-v1:0", "Claude Sonnet 4.5"),  # Best at complex reasoning
    ("meta.llama3-8b-instruct-v1:0", "Llama 3 8B"),                        # Open-weight, cost-efficient
    ("mistral.mistral-7b-instruct-v0:2", "Mistral 7B"),                    # Lightweight, fast inference
]

prompt = "What are the three most important considerations when choosing a foundation model for a production application?"

for model_id, name in models_to_compare:
    print(f"\n{'='*60}")
    print(f"Model: {name}")
    print(f"{'='*60}")
    try:
        output = invoke_converse(model_id, prompt)
        print(output)
    except Exception as e:
        print(f"Error: {e}")

## D. Parameter Tuning

Inference parameters control the style and structure of model output. Understanding these is critical for both the exam and real-world use.

| Parameter | Range | What It Controls | When to Use |
|-----------|-------|-----------------|-------------|
| **temperature** | 0.0--1.0 | Randomness of token selection | 0 for factual/classification, 0.7+ for creative |
| **top_p** | 0.0--1.0 | Nucleus sampling -- smallest token set with cumulative probability >= p | Alternative to temperature; tune one, not both |
| **max_tokens** | 1--model max | Hard ceiling on response length | Controls cost and truncation behavior |
| **stop sequences** | list of strings | Strings that halt generation when encountered | Structured output, preventing runaway responses |

### Temperature Comparison

Run this cell multiple times -- at temperature 0.0 you get the same result each time; at 1.0 each run varies.

In [ ]:
prompt = "Write a creative tagline for a cloud computing company."
model_id = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"

configs = [
    # temperature=0 → always picks the highest-probability token (deterministic)
    {"temperature": 0.0, "label": "temperature=0.0 (deterministic)"},
    {"temperature": 0.5, "label": "temperature=0.5 (balanced)"},
    # temperature=1 → flat probability distribution (maximum creativity / variance)
    {"temperature": 1.0, "label": "temperature=1.0 (creative)"},
]

for config in configs:
    response = bedrock_runtime.converse(
        modelId=model_id,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": 100, "temperature": config["temperature"]}
    )
    output = response["output"]["message"]["content"][0]["text"]
    print(f"\n{config['label']}:")
    print(f"  {output}")

### top_p Comparison

**top_p** (nucleus sampling) restricts the candidate token pool rather than scaling probabilities. A `top_p` of 0.1 considers only the top 10% of probability mass (very focused); 0.9 allows the top 90% (more variety). Typically you tune **either** temperature **or** top_p, not both.

In [ ]:
prompt = "List three innovative uses of generative AI in healthcare."
model_id = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"

top_p_values = [
    # topP=0.1 → only tokens in the top 10% of probability mass (very focused)
    {"topP": 0.1, "label": "top_p=0.1 (very focused)"},
    {"topP": 0.5, "label": "top_p=0.5 (balanced)"},
    # topP=0.9 → tokens in the top 90% of probability mass (diverse)
    {"topP": 0.9, "label": "top_p=0.9 (diverse)"},
]

for config in top_p_values:
    response = bedrock_runtime.converse(
        modelId=model_id,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        # When tuning top_p, leave temperature at its default to isolate the effect
        inferenceConfig={"maxTokens": 200, "topP": config["topP"]}
    )
    output = response["output"]["message"]["content"][0]["text"]
    print(f"\n{config['label']}:")
    print(f"  {output[:300]}...")

### max_tokens Effect

`max_tokens` sets a hard ceiling on generation length. Too low truncates mid-sentence (`stopReason: max_tokens`); too high increases cost without improving quality. Typical values: chatbot ~512, summarizer ~2048.

In [ ]:
prompt = "Explain the differences between supervised, unsupervised, and reinforcement learning."
model_id = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"

token_limits = [50, 150, 500]

for limit in token_limits:
    response = bedrock_runtime.converse(
        modelId=model_id,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        # temperature=0.3 for factual explanation — low randomness
        inferenceConfig={"maxTokens": limit, "temperature": 0.3}
    )
    output = response["output"]["message"]["content"][0]["text"]
    # stopReason tells you WHY generation stopped: "max_tokens" = truncated, "end_turn" = natural end
    stop_reason = response["stopReason"]
    print(f"\nmax_tokens={limit} (stop reason: {stop_reason}):")
    print(f"  {output[:200]}{'...' if len(output) > 200 else ''}")
    print(f"  [Response length: {len(output)} characters]")

## E. Streaming Responses

**Streaming** returns tokens incrementally as they are produced, reducing perceived latency from seconds to milliseconds for the first token. `ConverseStream` is the streaming version of Converse.

Key event types in the stream:
- **`contentBlockDelta`** -- a chunk of generated text
- **`messageStop`** -- generation complete (includes stop reason)
- **`metadata`** -- token usage statistics for cost tracking

In [ ]:
# ConverseStream — streaming version of Converse for real-time token delivery
response = bedrock_runtime.converse_stream(
    modelId="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    messages=[{"role": "user", "content": [{"text": "Explain RAG in 3 sentences."}]}],
    inferenceConfig={"maxTokens": 256}
)

print("Streaming response:")
for event in response["stream"]:
    if "contentBlockDelta" in event:
        # Each delta contains a small chunk of text — print without newline for streaming effect
        print(event["contentBlockDelta"]["delta"]["text"], end="", flush=True)
    if "messageStop" in event:
        print(f"\n\nStop reason: {event['messageStop']['stopReason']}")
    if "metadata" in event:
        # Token usage arrives in the final metadata event — use for cost tracking
        usage = event["metadata"]["usage"]
        print(f"Tokens — input: {usage['inputTokens']}, output: {usage['outputTokens']}")

## Key Takeaways

1. Bedrock provides access to multiple foundation model providers through a single service
2. InvokeModel uses model-specific request formats; Converse API provides a unified interface
3. Temperature controls randomness (0=deterministic, 1=creative); top_p controls nucleus sampling
4. Streaming responses enable real-time user experiences with lower perceived latency
5. Model selection depends on task requirements: capability, latency, cost, and context window size

## Key Concepts

| Concept | Definition |
|---------|-----------|
| `InvokeModel` | Bedrock API that sends a prompt to a model using provider-specific request format |
| `Converse` | Unified Bedrock API that works with all models using a consistent message format |
| `ConverseStream` | Streaming version of Converse that returns response tokens incrementally |
| Temperature | Controls randomness of output; 0 = deterministic, 1 = maximum creativity |
| `top_p` | Nucleus sampling parameter; limits token selection to the smallest set with cumulative probability >= p |
| Provisioned Throughput | Reserved model capacity for predictable performance; charged hourly regardless of usage |
| On-Demand | Pay-per-token pricing with no reserved capacity; subject to throttling under high load |

## Exam Preparation — Common Exam Question Patterns

**Q: Which API provides a consistent interface across all Bedrock models?**
A: The Converse API. It uses a unified message format that works with all supported models, unlike InvokeModel which requires model-specific request bodies.

**Q: When should you use provisioned throughput instead of on-demand?**
A: When you need guaranteed latency and throughput for production workloads. Provisioned throughput reserves dedicated capacity, eliminating throttling risks — but you pay hourly regardless of usage.

**Q: What happens when you set temperature to 0?**
A: The model produces deterministic, most-likely outputs. Running the same prompt multiple times will return the same (or nearly the same) response.

**Q: Which Bedrock API requires model-specific request formatting?**
A: InvokeModel. Each provider (Anthropic, Amazon, Meta, Mistral) has its own JSON schema for both the request body and response format.

**Q: How do you reduce perceived latency for end users?**
A: Use streaming (ConverseStream) to return tokens as they are generated. The user sees the response building in real time instead of waiting for the full generation to complete.

## Cost Breakdown

| Resource | Usage | Est. Cost |
|----------|-------|-----------|
| Bedrock API (Claude Sonnet 4.5) | ~5K input + 5K output tokens | ~$0.30 |
| Bedrock API (Llama 3 8B) | ~2K input + 2K output tokens | ~$0.01 |
| Bedrock API (Mistral 7B) | ~2K input + 2K output tokens | ~$0.01 |
| **Total** | | **~$0.35** |